<a href="https://colab.research.google.com/github/Amon127/IA-e-Machine-Learning/blob/main/Busca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import copy

class Noh:
    def __init__(self, estado, h, profundidade, pai=None):
        self.estado = estado
        self.h = h
        self.profundidade = profundidade
        self.pai = pai # Added parent attribute

    def troca(self, estado, x, y, novo_x, novo_y):
      tamanho_estado = len(self.estado)
      if (novo_x >= 0 and novo_x < tamanho_estado and novo_y >= 0 and novo_y < tamanho_estado):
        novo_estado = copy.deepcopy(estado)
        temp  = novo_estado[novo_x][novo_y]
        novo_estado[novo_x][novo_y] = novo_estado[x][y]
        novo_estado[x][y] = temp
        return novo_estado
      else:
          return None

    def encontrar(self, estado, x):
      tamanho_estado = len(self.estado)
      for i in range(0, tamanho_estado):
        for j in range(0, tamanho_estado):
          if estado[i][j] == x:
            return i,j

    def gerar_filhos(self):
        x, y = self.encontrar(self.estado, 0)
        lista_poscoes = [[x,y-1],[x,y+1],[x-1,y],[x+1,y]]
        filhos = []
        for posicao in lista_poscoes:
          estado_filho = self.troca(self.estado, x, y, posicao[0], posicao[1])
          if estado_filho is not None:
            # Pass self as the parent to the new Noh object
            noh_filho = Noh(estado_filho, 0, self.profundidade+1, pai=self)
            filhos.append(noh_filho)
        return filhos

In [ ]:
class Jogo:
    def __init__(self, tamanho):
      self.n = tamanho
      self.open = []
      self.closed = [] # Now stores Noh objects
      self.start = []
      self.goal = []
      self.goal_positions = {}

    def inicio(self):
      #caso de problema: 1 2 3,   0 8 5,   4 7 6
      self.start = [[1, 2, 3], [0, 8, 5], [4, 7, 6]]
      self.goal = [[1, 2, 3], [4, 5, 6], [7, 8, 0]]
      # Pre-compute goal positions for Manhattan distance
      for r in range(self.n):
          for c in range(self.n):
              self.goal_positions[self.goal[r][c]] = (r, c)

    def exibe(self, estado):
        print()
        for i in estado:
            for j in i:
                print(j, end=" ")
            print()

    def h(self, estado_atual):
        # Calculate Manhattan distance heuristic
        manhattan_distance = 0
        for r in range(self.n):
            for c in range(self.n):
                tile = estado_atual[r][c]
                if tile != 0: # We don't calculate distance for the empty tile (0)
                    goal_r, goal_c = self.goal_positions[tile]
                    manhattan_distance += abs(r - goal_r) + abs(c - goal_c)
        return manhattan_distance


    def busca(self):
        noh_inicial = Noh(self.start, 0, 0)
        noh_inicial.h = self.h(noh_inicial.estado)
        self.open.append(noh_inicial)

        while True:
          if not self.open:
              print('No solution found!')
              break

          noh_atual = self.open[0]
          del self.open[0] # Remove from open list

          # Add current node to closed list. Store the Noh object.
          self.closed.append(noh_atual)

          print('------------------')
          self.exibe(noh_atual.estado)
          print('h: ', noh_atual.h)
          print('Profundidade:', noh_atual.profundidade)

          if noh_atual.h == 0:
            print('Goal reached!')
            # Reconstruct and print the path
            path = []
            current = noh_atual
            while current:
                path.append(current.estado)
                current = current.pai
            path.reverse()
            print('\nSolution Path:')
            for step in path:
                self.exibe(step)
                print()
            break

          filhos = noh_atual.gerar_filhos()

          for filho in filhos:
            # Check if child is already in open or closed list (by state)
            if not any(self._compare_states(filho.estado, n.estado) for n in self.open) and \
               not any(self._compare_states(filho.estado, n.estado) for n in self.closed):
                filho.h = self.h(filho.estado)
                self.open.append(filho)

          self.open.sort(key = lambda x: x.h, reverse=False )

    def _compare_states(self, state1, state2):
        # Helper to compare two 2D lists (states)
        return state1 == state2

In [ ]:
def main():
    jogo8 = Jogo(3)
    jogo8.inicio()
    jogo8.busca()

main()

------------------

1 2 3 
0 8 5 
4 7 6 
h:  5
Profundidade: 0
------------------

1 2 3 
4 8 5 
0 7 6 
h:  4
Profundidade: 1
------------------

1 2 3 
4 8 5 
7 0 6 
h:  3
Profundidade: 2
------------------

1 2 3 
4 0 5 
7 8 6 
h:  2
Profundidade: 3
------------------

1 2 3 
4 5 0 
7 8 6 
h:  1
Profundidade: 4
------------------

1 2 3 
4 5 6 
7 8 0 
h:  0
Profundidade: 5
Goal reached!

Solution Path:

1 2 3 
0 8 5 
4 7 6 


1 2 3 
4 8 5 
0 7 6 


1 2 3 
4 8 5 
7 0 6 


1 2 3 
4 0 5 
7 8 6 


1 2 3 
4 5 0 
7 8 6 


1 2 3 
4 5 6 
7 8 0 

